# ERA5 AMJJA cut-off / deep-trough catalogue (1940–2025)

Runs the **exact colindex2 detector of this study** over every AMJJA day (Apr–Aug) and
saves **every detected feature** — closed cut-off (`ex==1`) and open deep trough (`ex==0`) —
with its `So`, location, radius and orientation, inside the **large classification box**
(lon −35…0, lat 25…50 N), to a single netCDF.

Detection is run over the wide detection region (lon −50…55, lat 5…90 N) as in the study,
then features are kept if they fall in the large box. Derived from
`ANALOGUES_composite_quality_factual_vs_counterfactual-6.ipynb` (same paths, Z500 file,
detector parameters and parallelisation). Purpose: multi-decadal trend of COL activity.

In [ ]:
import os
# --- cap threads PER PROCESS so N detection workers x M threads don't oversubscribe the node ---
# MUST be the first thing that runs, before numpy / numba / colindex2 are imported.
for _tv in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "NUMBA_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_tv, "1")
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # concurrent reads of the shared Z500 file

import sys, json, hashlib
from pathlib import Path
from contextlib import contextmanager
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# ============================ CONFIG (must match the study) ============================
path_project = "${ANALOGUE_BASE}/"
path_ERA5    = os.path.join(path_project, "rawdat/")
COL_Z500_FILE = os.path.join(path_ERA5, "era5_z500_daily_NA_1940_2026_FULL_1deg.nc")
COL_INPUT_IS_GEOPOTENTIAL = True                      # ERA5 z is m2 s-2 -> /9.80665

# --- season + years -------------------------------------------------------------------
MIN_MONTH, MAX_MONTH = 4, 8                           # AMJJA
YEAR_START, YEAR_END = 1940, 2025

# --- detection region (colindex2 runs here) and LARGE classification box (features kept here)
COL_DETECTION_LON_MIN, COL_DETECTION_LON_MAX = -50, 55
COL_DETECTION_LAT_MIN, COL_DETECTION_LAT_MAX = 5, 90
COL_LON_MIN, COL_LON_MAX = -35, 0                     # large box (study COL-activity region)
COL_LAT_MIN, COL_LAT_MAX = 25, 50

# --- detector parameters (identical to the study) -------------------------------------
COL_RADII_KM = np.arange(200, 2101, 100)
COL_DETECTION_SO_MIN = 3.0                            # So floor written to disk by Detect
COL_SR_MAX = 3.0
COL_REMOVE_MIN_RADIUS = COL_REMOVE_MAX_RADIUS = True
COL_USE_GPU = False; COL_CHUNK_SIZE = 100_000

# --- which features to SAVE ------------------------------------------------------------
#   both cut-offs (ex==1) and deep troughs (ex==0) are saved; the study COL-activity
#   qualifier is So>=6 (applied in the trend preview). Save everything >= floor so you can
#   test threshold sensitivity later.
SAVE_SO_MIN = COL_DETECTION_SO_MIN                    # = 3.0 (set to 6.0 to store only study-qualifying features)

# --- outputs + parallelisation ---------------------------------------------------------
OUT_DIR = os.path.join(path_project, "col_catalogue_AMJJA"); os.makedirs(OUT_DIR, exist_ok=True)
OUT_NC          = os.path.join(OUT_DIR, f"col_features_AMJJA_{YEAR_START}_{YEAR_END}_largebox.nc")
PERDAY_DIR = os.path.join(OUT_DIR, "per_day_features")             # one small csv per detected day
os.makedirs(PERDAY_DIR, exist_ok=True)                             # -> per-DATE resumability (survives any kill)
DETECTION_WORK_DIR = Path(OUT_DIR) / "colindex2" / "detection_workers"
STENCIL_ROOT       = Path(OUT_DIR) / "colindex2"

try:
    _cores = len(os.sched_getaffinity(0))
except AttributeError:
    _cores = os.cpu_count() or 1
N_DETECTION_WORKERS = 64#_cores        # each worker is now single-threaded -> scale to the core count
DETECTION_CHUNK_SIZE = 10          # dates per worker task (smaller = better tail load-balancing)

print("Z500:", COL_Z500_FILE)
print(f"season months {MIN_MONTH}-{MAX_MONTH} | years {YEAR_START}-{YEAR_END}")
print(f"detection region lon[{COL_DETECTION_LON_MIN},{COL_DETECTION_LON_MAX}] lat[{COL_DETECTION_LAT_MIN},{COL_DETECTION_LAT_MAX}]")
print(f"save features in LARGE box lon[{COL_LON_MIN},{COL_LON_MAX}] lat[{COL_LAT_MIN},{COL_LAT_MAX}] with So>={SAVE_SO_MIN}, SR<={COL_SR_MAX}")
print("cores allocated:", _cores, "-> detection workers:", N_DETECTION_WORKERS)
print("output netCDF:", OUT_NC)

In [ ]:
# ---- coordinate helpers (verbatim from the study notebook) ----
def squeeze_auxiliary_dimensions(data):
    """Keep only time/lat/lon dimensions."""
    for dim in list(data.dims):
        if dim not in ("time", "lat", "lon"):
            if data.sizes[dim] != 1:
                raise ValueError(f"Unexpected non-singleton dimension {dim}: {data.sizes[dim]}")
            data = data.isel({dim: 0}, drop=True)
    aux = [c for c in data.coords if c not in data.dims and c != "time"]
    if aux:
        data = data.drop_vars(aux)
    return data

def select_latitude_range(field, minimum, maximum):
    """Select latitude bounds for ascending or descending coordinates."""
    if bool(field["lat"][0] < field["lat"][-1]):
        return field.sel(lat=slice(minimum, maximum))
    return field.sel(lat=slice(maximum, minimum))

In [ ]:
# ---- colindex2 detector (identical settings to the study) ----
try:
    from colindex2 import Detect
except ImportError as exc:
    raise ImportError("colindex2 not installed in this kernel (`%pip install colindex2`).") from exc

@contextmanager
def colindex_working_directory(path):
    previous = Path.cwd(); path = Path(path); path.mkdir(parents=True, exist_ok=True); os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)

def run_z500_detector(height, output_directory, stencil_directory):
    if "time" in height.dims and height.sizes["time"] == 1:
        height = height.isel(time=0)
    with colindex_working_directory(stencil_directory):
        Detect(height, odir=str(output_directory), ty="L", r=COL_RADII_KM,
               SR_thres=COL_SR_MAX, So_thres=COL_DETECTION_SO_MIN, Do_thres=0.0,
               xx_thres=0.0, rm_rmin=COL_REMOVE_MIN_RADIUS, rm_rmax=COL_REMOVE_MAX_RADIUS,
               local_ex_req_num=7, gpu=COL_USE_GPU, chunk_size=COL_CHUNK_SIZE,
               nc=True, subs=False)

def load_z500_for_detection(dates):
    src = xr.open_dataarray(COL_Z500_FILE)
    try:
        height = src.sel(time=pd.DatetimeIndex(dates)).load()
    finally:
        src.close()
    height = squeeze_auxiliary_dimensions(height)
    height = height.sel(lon=slice(COL_DETECTION_LON_MIN, COL_DETECTION_LON_MAX))
    height = select_latitude_range(height, COL_DETECTION_LAT_MIN, COL_DETECTION_LAT_MAX)
    if COL_INPUT_IS_GEOPOTENTIAL:
        height = height / 9.80665
    height = height.assign_attrs(units="m", standard_name="geopotential_height")
    return height.assign_coords(level=500).transpose("time", "lat", "lon")

def colindex_csv_path(output_directory, timestamp):
    ts = pd.Timestamp(timestamp)
    return Path(output_directory) / "V" / ts.strftime("%Y%m") / ("V-L-" + ts.strftime("%Y%m%d%H%M") + "-0500.csv")

FEATCOLS = ["date", "lon", "lat", "So", "ex", "up", "SR", "radius_km"]

def read_features(output_directory, timestamp):
    """All detected features inside the LARGE box for one date (both ex==1 and ex==0)."""
    csv_path = colindex_csv_path(output_directory, timestamp)
    if not csv_path.exists():
        raise FileNotFoundError(f"colindex2 output missing: {csv_path}")
    pts = pd.read_csv(csv_path)
    if pts.empty:
        return pd.DataFrame(columns=FEATCOLS)
    lon180 = (pts["lon"] + 180) % 360 - 180
    keep = (lon180.between(COL_LON_MIN, COL_LON_MAX)
            & pts["lat"].between(COL_LAT_MIN, COL_LAT_MAX)
            & pts["So"].ge(SAVE_SO_MIN))
    if not keep.any():
        return pd.DataFrame(columns=FEATCOLS)
    p = pts.loc[keep].copy()
    out = pd.DataFrame({
        "date":      pd.Timestamp(timestamp).normalize(),
        "lon":       lon180[keep].values,
        "lat":       p["lat"].values,
        "So":        p["So"].values,
        "ex":        pd.to_numeric(p["ex"], errors="coerce").fillna(0).astype(int).values,   # 1=cut-off, 0=deep trough
        "up":        (p["up_or_down"].astype(str) == "up").astype(int).values if "up_or_down" in p else 0,
        "SR":        p["SR"].values if "SR" in p else np.nan,
        "radius_km": p["ro"].values if "ro" in p else np.nan,
    })
    return out[FEATCOLS]

In [ ]:
# ---- all AMJJA dates present in the Z500 file ----
_z = xr.open_dataarray(COL_Z500_FILE)
_t = pd.DatetimeIndex(_z["time"].values); _z.close()
_mask = ((_t.month >= MIN_MONTH) & (_t.month <= MAX_MONTH)
         & (_t.year >= YEAR_START) & (_t.year <= YEAR_END))
AMJJA_DATES = _t[_mask].normalize().unique()
print(f"{len(AMJJA_DATES)} AMJJA days | {AMJJA_DATES.year.min()}-{AMJJA_DATES.year.max()}")

In [ ]:
# ---- parallel detection -> one csv PER DAY (resumable at day granularity, atomic) ----
def _perday_path(date):
    return os.path.join(PERDAY_DIR, pd.Timestamp(date).strftime("%Y%m%d") + ".csv")

def _detect_features_chunk(chunk):
    """Detect a chunk, then write each day's features to its own file immediately.
    Writing happens INSIDE the worker, per day, so a kill only loses in-flight days."""
    cid, cdates = chunk
    wb = Path(DETECTION_WORK_DIR) / f"worker_{cid:04d}"
    run_z500_detector(load_z500_for_detection(cdates), wb / "out", wb / "stencil_cache")
    for d in pd.DatetimeIndex(cdates):
        df = read_features(wb / "out", d)                 # may be empty (still marks the day done)
        tmp = _perday_path(d) + ".tmp"
        df.to_csv(tmp, index=False); os.replace(tmp, _perday_path(d))   # atomic
    return len(cdates)

def collect_features(dates):
    import multiprocessing as mp
    from concurrent.futures import ProcessPoolExecutor, as_completed
    wanted = pd.DatetimeIndex(pd.to_datetime(np.asarray(dates))).normalize().unique()

    done = {f[:-4] for f in os.listdir(PERDAY_DIR) if f.endswith(".csv")}          # 'YYYYMMDD' stems
    missing = pd.DatetimeIndex([d for d in wanted if d.strftime("%Y%m%d") not in done])
    if len(missing):
        chunks = [(i, list(missing[s:s + DETECTION_CHUNK_SIZE]))
                  for i, s in enumerate(range(0, len(missing), DETECTION_CHUNK_SIZE))]
        workers = max(1, int(N_DETECTION_WORKERS))
        print(f"Detecting {len(missing)} uncached days ({len(wanted) - len(missing)} already on disk) "
              f"in {len(chunks)} chunks on {workers} worker(s) ...")
        done_n = 0
        if workers == 1:
            for ch in chunks:
                done_n += _detect_features_chunk(ch); print(f"  {done_n}/{len(missing)}")
        else:
            ctx = mp.get_context("fork")   # workers inherit notebook globals + detector helpers
            with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as pool:
                futs = [pool.submit(_detect_features_chunk, ch) for ch in chunks]
                for fut in as_completed(futs):
                    done_n += fut.result(); print(f"  {done_n}/{len(missing)}")
    else:
        print("all requested days already on disk.")

    # aggregate the per-day files for the requested dates
    frames = []
    for d in wanted:
        p = _perday_path(d)
        if os.path.exists(p):
            fr = pd.read_csv(p, parse_dates=["date"])
            if len(fr):
                frames.append(fr)
    feats = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=FEATCOLS)
    return feats.sort_values("date").reset_index(drop=True) if len(feats) else feats

features = collect_features(AMJJA_DATES)
_nc = int((features.ex == 1).sum()) if len(features) else 0
_nt = int((features.ex == 0).sum()) if len(features) else 0
print(f"\ncollected {len(features)} features over {features['date'].nunique() if len(features) else 0} days "
      f"({_nc} cut-offs, {_nt} deep troughs)")
features.head()

In [ ]:
# ---- cut-off-low DAYS per year (MJ vs JA) + linear trends ----
from scipy import stats

SO_MIN = 6.0                                   # study cut-off-low qualifier (lower to explore sensitivity)

_c = xr.open_dataset(OUT_NC)                    # or: d = features.copy()  (skip the next 2 lines then)
d = _c[["time", "So", "ex"]].to_dataframe().reset_index(); _c.close()

d = d[(d["ex"] == 1) & (d["So"] >= SO_MIN)]     # closed cut-off lows only
d["date"]  = pd.DatetimeIndex(d["time"]).normalize()
d["year"]  = pd.DatetimeIndex(d["time"]).year
d["month"] = pd.DatetimeIndex(d["time"]).month

YEARS = np.arange(YEAR_START, YEAR_END + 1)
BINS  = {"MJ (May-Jun)": [5, 6], "JA (Jul-Aug)": [7, 8]}

def cutoff_days(months):                        # # of distinct days with >=1 cut-off low in the bin
    sub = d[d["month"].isin(months)]
    return sub.groupby("year")["date"].nunique().reindex(YEARS, fill_value=0).values

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
for ax, (lab, months) in zip(axes, BINS.items()):
    y = cutoff_days(months)
    ax.bar(YEARS, y, color="#c0392b", alpha=0.7, width=0.9)
    sl, ic, r, p, se = stats.linregress(YEARS, y)
    ax.plot(YEARS, ic + sl * YEARS, "k-", lw=2.3)
    ax.set_title(f"{lab}: cut-off-low days/yr  -  trend {sl*10:+.2f}/decade (p={p:.3f}, r={r:+.2f})", fontsize=11)
    ax.set_xlabel("year"); ax.grid(alpha=.3)
axes[0].set_ylabel(f"# days with a closed cut-off (ex=1, So>={SO_MIN:.0f}, large box)")
fig.suptitle("Cut-off-low day count - early (MJ) vs late (JA) summer", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "cutoff_days_MJ_JA.png"), dpi=140, bbox_inches="tight")
plt.show()

for lab, months in BINS.items():
    y = cutoff_days(months); sl, ic, r, p, se = stats.linregress(YEARS, y)
    print(f"{lab}: {sl*10:+.2f} days/decade (p={p:.3f}), mean {y.mean():.1f} days/yr")

In [ ]:
# ---- how saturated is the cut-off-day count? scan So thresholds ----
_c = xr.open_dataset(OUT_NC); cut = _c[["time","So","ex"]].to_dataframe().reset_index(); _c.close()
cut = cut[cut.ex == 1].copy()
cut["date"] = pd.DatetimeIndex(cut.time).normalize()
cut["year"] = pd.DatetimeIndex(cut.time).year
cut["month"] = pd.DatetimeIndex(cut.time).month
YEARS = np.arange(YEAR_START, YEAR_END + 1)

print("So percentiles of closed cut-offs:",
      dict(zip([50,75,90,95,99], np.percentile(cut.So, [50,75,90,95,99]).round(1))))
print(f"{'So>=':>6} {'MJ days/yr':>12} {'(% of 61)':>10} {'JA days/yr':>12} {'(% of 62)':>10}")
for thr in [6, 8, 10, 12, 15, 20]:
    row = [f"{thr:>6}"]
    for mo, nd in [([5,6], 61), ([7,8], 62)]:
        dpy = cut[(cut.So >= thr) & (cut.month.isin(mo))].groupby("year")["date"].nunique().reindex(YEARS, fill_value=0)
        row += [f"{dpy.mean():>12.1f}", f"{100*dpy.mean()/nd:>9.0f}%"]
    print(" ".join(row))

In [ ]:
# ---- write the netCDF (one 'feature' dimension; ragged catalogue) ----
f = features.reset_index(drop=True)
ds = xr.Dataset(
    data_vars={
        "time":      ("feature", pd.DatetimeIndex(f["date"]).values),
        "lon":       ("feature", f["lon"].astype("float32").values),
        "lat":       ("feature", f["lat"].astype("float32").values),
        "So":        ("feature", f["So"].astype("float32").values),
        "ex":        ("feature", f["ex"].astype("int8").values),
        "up":        ("feature", f["up"].astype("int8").values),
        "SR":        ("feature", f["SR"].astype("float32").values),
        "radius_km": ("feature", f["radius_km"].astype("float32").values),
    },
    coords={"feature": np.arange(len(f), dtype="int32")},
    attrs={
        "title": "ERA5 AMJJA cut-off / deep-trough catalogue (colindex2), large box",
        "detector": "colindex2 Detect ty=L, radii 200-2100 km, So_thres=3, SR_thres=3, rm_rmin/rmax=True, local_ex_req_num=7",
        "detection_region_lon": [COL_DETECTION_LON_MIN, COL_DETECTION_LON_MAX],
        "detection_region_lat": [COL_DETECTION_LAT_MIN, COL_DETECTION_LAT_MAX],
        "saved_box_lon": [COL_LON_MIN, COL_LON_MAX],
        "saved_box_lat": [COL_LAT_MIN, COL_LAT_MAX],
        "save_So_min": float(SAVE_SO_MIN),
        "season_months": [MIN_MONTH, MAX_MONTH],
        "years": [YEAR_START, YEAR_END],
        "ex": "1 = closed cut-off, 0 = open deep trough",
        "up": "1 = up (poleward reversal / 'upstream'), 0 = down",
        "So": "colindex2 strength; study COL-activity = sum(So) over So>=6",
        "source_z500": os.path.basename(COL_Z500_FILE),
    },
)
ds.to_netcdf(OUT_NC)
print("saved", OUT_NC, "|", len(f), "features")
ds

In [ ]:
# ---- preview: multi-decadal AMJJA COL-activity trend in the large box (study qualifier So>=6) ----
from scipy import stats
SO_ACTIVITY = 6.0
q = f[f["So"] >= SO_ACTIVITY].copy(); q["year"] = pd.DatetimeIndex(q["date"]).year
yrs = np.arange(YEAR_START, YEAR_END + 1)
series = {
    "closed cut-offs (ex=1)": q[q.ex == 1].groupby("year")["So"].sum().reindex(yrs, fill_value=0.0),
    "deep troughs (ex=0)":    q[q.ex == 0].groupby("year")["So"].sum().reindex(yrs, fill_value=0.0),
    "all (cut-off + trough)": q.groupby("year")["So"].sum().reindex(yrs, fill_value=0.0),
}
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6), sharex=True)
for ax, (lab, s) in zip(axes, series.items()):
    ax.bar(yrs, s.values, color="#6a8caf", alpha=0.65)
    sl, ic, r, p, se = stats.linregress(yrs, s.values)
    ax.plot(yrs, ic + sl * yrs, "k-", lw=2)
    ax.set_title(f"{lab}\ntrend {sl:+.2f} ΣSo/yr (p={p:.3f}, r={r:+.2f})", fontsize=11)
    ax.set_xlabel("year"); ax.grid(alpha=.3)
axes[0].set_ylabel(f"annual ΣSo  (So≥{SO_ACTIVITY:.0f}, large box)")
fig.suptitle(f"ERA5 AMJJA COL activity {YEAR_START}-{YEAR_END} (large box −35..0, 25..50N)", y=1.03, fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "col_activity_trend_AMJJA.png"), dpi=140, bbox_inches="tight")
plt.show()
for lab, s in series.items():
    sl, ic, r, p, se = stats.linregress(yrs, s.values)
    print(f"{lab:28s}: {sl:+.3f} ΣSo/yr  (p={p:.3f})  mean={s.mean():.1f}")

## Multi-decadal trend analysis

Reads the saved catalogue. Two-month bins **MJ** (May–Jun) and **JA** (Jul–Aug) plus **May** alone.
Per-year activity = `sum(So) / number of days in the bin` (mean So per day). Spatial trend =
per-grid-cell OLS slope of the seasonal `sum(So)`, expressed **per decade**, with p≤0.05 stippled.
`EX_FILTER` toggles cut-offs+troughs / cut-offs only (`ex==1`) / troughs only (`ex==0`).

In [ ]:
# ---- load catalogue + trend helpers ----
import cartopy.crs as ccrs, cartopy.feature as cfeature
from scipy import stats

_cat = xr.open_dataset(OUT_NC)
adf = _cat[["time","lon","lat","So","ex","up"]].to_dataframe().reset_index(); _cat.close()
adf["year"] = pd.DatetimeIndex(adf["time"]).year
adf["month"] = pd.DatetimeIndex(adf["time"]).month

SO_ACTIVITY = 6.0          # study COL-activity qualifier
EX_FILTER   = None         # None = cut-offs+troughs ; 1 = closed cut-offs only ; 0 = deep troughs only
Q = adf[adf["So"] >= SO_ACTIVITY]
if EX_FILTER is not None:
    Q = Q[Q["ex"] == EX_FILTER]
YEARS = np.arange(YEAR_START, YEAR_END + 1)

DLON = DLAT = 3.0          # trend-map grid (deg)
GLON = np.arange(COL_LON_MIN, COL_LON_MAX + 1e-6, DLON)
GLAT = np.arange(COL_LAT_MIN, COL_LAT_MAX + 1e-6, DLAT)
LONC = 0.5 * (GLON[:-1] + GLON[1:]); LATC = 0.5 * (GLAT[:-1] + GLAT[1:])

def _ndays(year, months):
    dr = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
    return int(dr.month.isin(months).sum())

def yearly_mean_So(months):
    """Per-year sum(So) over the bin, divided by number of calendar days in the bin."""
    s = Q[Q["month"].isin(months)].groupby("year")["So"].sum().reindex(YEARS, fill_value=0.0).values
    nd = np.array([_ndays(y, months) for y in YEARS], float)
    return s / nd

def cell_trend(months):
    """Per-grid-cell OLS trend (per decade) of the seasonal sum(So), with p-value."""
    stack = np.zeros((len(YEARS), len(LATC), len(LONC)))
    for i, y in enumerate(YEARS):
        sub = Q[(Q.year == y) & (Q.month.isin(months))]
        H, _, _ = np.histogram2d(sub.lon.values, sub.lat.values, bins=[GLON, GLAT], weights=sub.So.values)
        stack[i] = H.T
    n = len(YEARS); x = (YEARS - YEARS.mean()).astype(float); Sxx = (x ** 2).sum()
    slope = (stack * x[:, None, None]).sum(0) / Sxx
    resid = stack - (slope[None] * x[:, None, None] + stack.mean(0)[None])
    se = np.sqrt((resid ** 2).sum(0) / (n - 2) / Sxx)
    with np.errstate(invalid="ignore", divide="ignore"):
        t = slope / se
    p = 2 * stats.t.sf(np.abs(t), n - 2)
    return slope * 10.0, p     # per decade

print(f"analysis ready | qualifying features (So>={SO_ACTIVITY:.0f}, EX_FILTER={EX_FILTER}):", len(Q))

In [ ]:
# ---- Fig 1: MJ vs JA per-year mean So/day + linear trend ----
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, (lab, months) in zip(axes, [("MJ (May-Jun)", [5, 6]), ("JA (Jul-Aug)", [7, 8])]):
    y = yearly_mean_So(months)
    ax.bar(YEARS, y, color="#6a8caf", alpha=0.75, width=0.9)
    sl, ic, r, p, se = stats.linregress(YEARS, y)
    ax.plot(YEARS, ic + sl * YEARS, "r-", lw=2.3)
    ax.set_title(f"{lab}: mean So/day  -  trend {sl*10:+.3f}/decade (p={p:.3f}, r={r:+.2f})", fontsize=11)
    ax.set_xlabel("year"); ax.grid(alpha=.3)
axes[0].set_ylabel(f"seasonal mean So per day  (So>={SO_ACTIVITY:.0f}, large box)")
fig.suptitle("AMJJA COL activity - early (MJ) vs late (JA) summer", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "trend_hist_MJ_JA.png"), dpi=140, bbox_inches="tight"); plt.show()

In [ ]:
# ---- Fig 2: MJ & JA spatial trend maps ( SumSo per decade; stipple p<=0.05) ----
proj = ccrs.PlateCarree()
res = {lab: cell_trend(m) for lab, m in [("MJ (May-Jun)", [5, 6]), ("JA (Jul-Aug)", [7, 8])]}
vmax = np.nanpercentile(np.abs(np.concatenate([res[k][0].ravel() for k in res])), 98) or 1.0
fig, axes = plt.subplots(1, 2, figsize=(15, 6), subplot_kw={"projection": proj}, constrained_layout=True)
pm = None
for ax, lab in zip(axes, res):
    sl, p = res[lab]
    ax.set_extent([COL_LON_MIN, COL_LON_MAX, COL_LAT_MIN, COL_LAT_MAX], crs=proj)
    pm = ax.pcolormesh(LONC, LATC, sl, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto", transform=proj)
    ax.contourf(LONC, LATC, p, levels=[0, 0.05], hatches=["...."], colors="none", transform=proj)
    ax.add_feature(cfeature.COASTLINE, lw=0.6); ax.add_feature(cfeature.BORDERS, lw=0.3)
    gl = ax.gridlines(draw_labels=True, ls="--", alpha=0.3); gl.top_labels = gl.right_labels = False
    ax.set_title(f"{lab} - SumSo trend per decade")
fig.colorbar(pm, ax=axes, location="bottom", shrink=0.6, aspect=40, pad=0.05,
             label="SumSo trend per decade  (stipple p<=0.05)")
fig.savefig(os.path.join(OUT_DIR, "trend_map_MJ_JA.png"), dpi=140, bbox_inches="tight"); plt.show()

In [ ]:
# ---- Fig 3: MAY only - histogram + spatial trend map ----
fig = plt.figure(figsize=(15, 5.2))
ax1 = fig.add_subplot(1, 2, 1)
y = yearly_mean_So([5]); sl, ic, r, p, se = stats.linregress(YEARS, y)
ax1.bar(YEARS, y, color="#6a8caf", alpha=0.75, width=0.9); ax1.plot(YEARS, ic + sl * YEARS, "r-", lw=2.3)
ax1.set_title(f"May: mean So/day  -  trend {sl*10:+.3f}/decade (p={p:.3f}, r={r:+.2f})", fontsize=11)
ax1.set_xlabel("year"); ax1.set_ylabel(f"mean So per day (So>={SO_ACTIVITY:.0f})"); ax1.grid(alpha=.3)

ax2 = fig.add_subplot(1, 2, 2, projection=proj)
sl_m, p_m = cell_trend([5]); vmax_m = np.nanpercentile(np.abs(sl_m), 98) or 1.0
ax2.set_extent([COL_LON_MIN, COL_LON_MAX, COL_LAT_MIN, COL_LAT_MAX], crs=proj)
pm = ax2.pcolormesh(LONC, LATC, sl_m, cmap="RdBu_r", vmin=-vmax_m, vmax=vmax_m, shading="auto", transform=proj)
ax2.contourf(LONC, LATC, p_m, levels=[0, 0.05], hatches=["...."], colors="none", transform=proj)
ax2.add_feature(cfeature.COASTLINE, lw=0.6); ax2.add_feature(cfeature.BORDERS, lw=0.3)
gl = ax2.gridlines(draw_labels=True, ls="--", alpha=0.3); gl.top_labels = gl.right_labels = False
ax2.set_title("May - SumSo trend per decade")
fig.colorbar(pm, ax=ax2, location="bottom", shrink=0.7, aspect=30, pad=0.06, label="SumSo trend per decade (stipple p<=0.05)")
fig.suptitle("May-only COL activity trend", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "trend_May.png"), dpi=140, bbox_inches="tight"); plt.show()